# Day 2 · 07 — Lab: model selection and ensembling, then final standings (20 + 5 min)

Every lab saved its forecasts to `results/day2/forecasts/`. Here we stop asking "which model has the lowest
mean RMSSE?" and ask better questions:

1. **Win rates**: on what share of series does model A beat model B?
2. **Elo and Bradley–Terry ratings**: one strength number per model from all pairwise series-level duels.
3. **Performance by volume and shape**: velocity class (A–J), demand type (smooth / erratic / intermittent /
   lumpy) and the precomputed shape clusters in `series_features.parquet`.
4. **Ensembles**: mean, median, top-k, and per-segment selection, chosen *without* peeking at the test score.
5. **Final standings.**

In [1]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import polars as pl

import bootcamp as bc

warnings.filterwarnings("ignore")
palette = ["#000000", "#0072B2", "#D55E00", "#009E73", "#CC79A7", "#56B4E9", "#E69F00"]
plt.rcParams.update({"figure.figsize": (12, 5), "axes.prop_cycle": plt.cycler(color=palette)})

panel = bc.load_panel()
train, test = bc.split(panel)
fc = bc.load_forecasts()
fc = fc[fc.unique_id.isin(test["unique_id"].unique())]
fc = fc.drop(columns=[c for c in fc.columns if c.startswith(("ENS_", "SEL_"))])   # from a previous run of this lab
coverage = fc.groupby("unique_id")[bc.model_columns(fc)].apply(lambda g: g.notna().all()).mean()
models = list(coverage[coverage >= 0.99].index)    # e.g. foundation models run on a CPU subset are left out
if len(models) < len(coverage):
    print("left out (forecast only a subset of series):", list(coverage[coverage < 0.99].index))
fc = fc[["unique_id", "ds"] + models].dropna(subset=models, how="any")
print(f"{len(models)} models: {models}")
print(f"{fc.unique_id.nunique():,} series with a forecast from every model")

/home/user/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


19 models: ['Naive', 'SeasonalNaive', 'AutoETS', 'LightGBM', 'NHITS', 'NBEATS_generic', 'NBEATS_interpretable', 'DLinear', 'NHITS_tuned', 'NHITS_noexog', 'NHITSx', 'LightGBMx', 'LightGBMx_mono', 'LightGBM_noexog', 'ETS_TopDown', 'ETS_MinT-wls', 'ETS_MinT-shrink', 'NHITS_MQ', 'LightGBM_conf']
1,997 series with a forecast from every model


In [2]:
long = bc.score(fc, test, train, models)
R = long[long.metric == "rmsse"].set_index("unique_id")[models].dropna()
print(R.shape)
R.mean().sort_values().round(3)

(1996, 19)


NHITS_noexog            1.101
NHITSx                  1.119
NHITS_MQ                1.120
LightGBM                1.131
NHITS_tuned             1.142
NHITS                   1.159
LightGBMx               1.165
LightGBMx_mono          1.166
NBEATS_generic          1.188
NBEATS_interpretable    1.192
ETS_MinT-shrink         1.251
AutoETS                 1.253
LightGBM_conf           1.286
LightGBM_noexog         1.293
Naive                   1.300
ETS_TopDown             1.320
DLinear                 1.376
SeasonalNaive           1.486
ETS_MinT-wls            1.813
dtype: float64

## Intuition: a league table for models

![Rating difference vs win chance](slides/img/elo_curve.png)

Treat every series as a **match** between two models: whoever has the lower RMSSE wins. The **win rate**
ignores by how much, so one disaster cannot dominate. **Elo / Bradley–Terry** turn all the matches into one
rating, as in chess: +100 points ≈ 64 % chance to win a series. An **ensemble** is a panel of forecasters
whose mistakes partly cancel.

## 1. Pairwise win rates

Cell (row A, column B) = share of series where A has a lower RMSSE than B. Ties (identical errors, e.g. two
models that both forecast zero) count half.

In [3]:
def win_matrix(R: pd.DataFrame) -> pd.DataFrame:
    a = R.values
    W = (a[:, :, None] < a[:, None, :]).mean(0) + 0.5 * (a[:, :, None] == a[:, None, :]).mean(0)
    np.fill_diagonal(W, np.nan)
    return pd.DataFrame(W, index=R.columns, columns=R.columns)

W = win_matrix(R)
order = W.mean(axis=1).sort_values(ascending=False).index
W = W.loc[order, order]
fig, ax = plt.subplots(figsize=(1 + 0.7 * len(models), 0.6 * len(models)))
im = ax.imshow(W, cmap="RdBu", vmin=0.2, vmax=0.8)
ax.set_xticks(range(len(order)), order, rotation=60, ha="right"), ax.set_yticks(range(len(order)), order)
for i in range(len(order)):
    for j in range(len(order)):
        if i != j:
            ax.text(j, i, f"{W.iat[i, j]:.2f}", ha="center", va="center", fontsize=8)
plt.colorbar(im, label="row beats column")
ax.set_title("Pairwise win rate (series level, RMSSE)")
plt.tight_layout()

## 2. Elo and Bradley–Terry

* **Bradley–Terry**: P(A beats B) = s_A / (s_A + s_B). Fit the strengths s by maximum likelihood on all
  pairwise wins (the classic MM algorithm, ~20 lines below).
* **Elo**: replay every series as a round-robin tournament in random order and update ratings after each duel.
  Order-dependent, so we average over shuffles. Popular in LLM arenas, and the same idea.

In [4]:
def bradley_terry(R: pd.DataFrame, iters: int = 500) -> pd.Series:
    a = R.values
    wins = (a[:, :, None] < a[:, None, :]).sum(0) + 0.5 * (a[:, :, None] == a[:, None, :]).sum(0)
    np.fill_diagonal(wins, 0)
    games = wins + wins.T
    s = np.ones(a.shape[1])
    for _ in range(iters):
        denom = (games / (s[:, None] + s[None, :])).sum(1)
        s_new = wins.sum(1) / denom
        s_new /= np.exp(np.log(s_new).mean())
        if np.allclose(s, s_new, rtol=1e-10):
            break
        s = s_new
    return pd.Series(400 * np.log10(s) + 1000, index=R.columns, name="bradley_terry")  # Elo-like scale


def elo(R: pd.DataFrame, k: float = 4.0, n_shuffles: int = 10, seed: int = 0) -> pd.Series:
    rng = np.random.default_rng(seed)
    a, m = R.values, R.shape[1]
    pairs = [(i, j) for i in range(m) for j in range(i + 1, m)]
    out = np.zeros(m)
    for _ in range(n_shuffles):
        r = np.full(m, 1000.0)
        for row in a[rng.permutation(len(a))]:
            for i, j in pairs:
                e = 1 / (1 + 10 ** ((r[j] - r[i]) / 400))
                s = 1.0 if row[i] < row[j] else 0.0 if row[i] > row[j] else 0.5
                r[i] += k * (s - e)
                r[j] -= k * (s - e)
        out += r
    return pd.Series(out / n_shuffles, index=R.columns, name="elo")


ratings = pd.concat([bradley_terry(R), elo(R), R.mean().rename("mean_rmsse"),
                     W.mean(axis=1).rename("avg_win_rate")], axis=1).sort_values("bradley_terry", ascending=False)
ratings.round(3)

,bradley_terry,elo,mean_rmsse,avg_win_rate
NHITSx,1130.927,1135.063,1.119,0.681
LightGBM,1089.578,1091.514,1.131,0.626
NHITS_noexog,1075.793,1090.191,1.101,0.607
LightGBMx,1064.564,1060.630,1.165,0.591
LightGBMx_mono,1060.662,1062.882,1.166,0.586
NHITS_MQ,1056.114,1059.602,1.120,0.579
NHITS_tuned,1046.223,1042.260,1.142,0.565
NHITS,1037.673,1055.540,1.159,0.553
NBEATS_generic,1036.663,1051.655,1.188,0.551
NBEATS_interpretable,1035.702,1052.779,1.192,0.550


Do the rating order and the mean-RMSSE order agree? When they disagree, one model is winning **often but by
little**, the other **rarely but by a lot** on a few series (which drives the mean).

### Reading the result

In the reference run the two orders disagree at the top. **NHITSx** is 2nd on mean RMSSE but 1st by
Bradley–Terry (it wins 68 % of its duels), and **LightGBM** is 4th on the mean but 2nd by Bradley–Terry.
**NHITS_noexog** is 1st on the mean but only 3rd in the duels. The reason is the ~340 intermittent series:
17 % of the panel, but 40–46 % of the summed RMSSE. NHITSx and LightGBM win many ordinary series by a little
and lose some intermittent ones by a lot, so the mean is mostly a contest about the hardest items.

*Reference run: 2,000 series, laptop CPU, 500 steps. Your numbers will differ a little; the reading should not.*

## 3. Performance by volume and shape

In [5]:
info = bc.series_info(panel).to_pandas().set_index("unique_id")
Rx = R.join(info)
best = Rx[models].idxmin(axis=1).rename("best_model")

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
by_class = Rx.groupby("sale_class")[models].mean()
by_class[order[:6]].plot(marker="o", ax=axes[0])
axes[0].set_title("Mean RMSSE by velocity class (A = fastest)"), axes[0].set_ylabel("RMSSE")
pd.crosstab(Rx["demand_type"], best, normalize="index").reindex(columns=order, fill_value=0).plot.barh(stacked=True, ax=axes[1], legend=False)
axes[1].set_title("Which model wins, by demand type"), axes[1].set_xlabel("share of series")
axes[1].legend(bbox_to_anchor=(1.01, 1), fontsize=8)
plt.tight_layout()

In [6]:
Rx.groupby("demand_type")[list(order[:6])].mean().round(3).assign(n=Rx.groupby("demand_type").size())

,NHITSx,LightGBM,NHITS_noexog,LightGBMx,LightGBMx_mono,NHITS_MQ,n
demand_type,,,,,,,
erratic,0.724,0.783,0.878,0.885,0.882,0.908,287
intermittent,3.021,2.823,2.587,2.853,2.849,2.623,337
lumpy,1.332,1.512,1.397,1.556,1.554,1.388,187
smooth,0.639,0.674,0.686,0.691,0.694,0.702,1185


**Reading it.** No model wins everywhere. In the reference run NHITSx is best on **smooth** (≈ 0.64),
**erratic** (≈ 0.72) and **lumpy** (≈ 1.33) series, where promotions and price moves explain the spikes. On
**intermittent** series (mostly zeros) the model *without* covariates wins (NHITS_noexog ≈ 2.59 vs NHITSx ≈ 3.0):
there the covariates mostly give the network more ways to predict a spike that does not come. Errors on
intermittent series are 4× larger than anywhere else, whatever the model.

The cluster table below comes from Day 1's feature clustering. Several clusters hold only 2–6 series, so read
only the big ones.

In [7]:
Rx.groupby("cluster_label")[list(order[:6])].mean().round(3).assign(n=Rx.groupby("cluster_label").size())

,NHITSx,LightGBM,NHITS_noexog,LightGBMx,LightGBMx_mono,NHITS_MQ,n
cluster_label,,,,,,,
0,6.443,6.218,5.456,6.232,6.191,5.537,152
1,0.668,0.683,0.754,0.730,0.731,0.764,952
3,1.039,1.407,1.180,1.337,1.277,1.159,2
5,0.348,0.357,0.361,0.412,0.438,0.485,3
6,0.404,0.341,0.579,0.523,0.495,0.665,5
9,0.573,0.629,0.536,0.644,0.661,0.548,382
10,0.643,0.667,0.827,0.719,0.713,0.842,216
11,1.016,1.065,1.217,1.623,1.706,1.086,2
12,0.611,0.778,0.713,0.755,0.747,0.926,26


## 4. Ensembles

Choosing on the test window and then scoring on it is cheating. To stay honest without extra training runs we
use a **cross-sectional split**: half the series ("selection set") decide which models go in, the other half
("evaluation set") is scored. With more compute, the proper version uses a validation *time* window
(`bc.VALID_START`) for every model.

In [8]:
rng = np.random.default_rng(bc.RANDOM_SEED)
ids = R.index.to_numpy()
sel_ids = set(rng.choice(ids, size=len(ids) // 2, replace=False))
is_sel = R.index.isin(sel_ids)
R_sel, info_sel = R[is_sel], Rx[is_sel]

top3 = list(R_sel.mean().nsmallest(3).index)
seg_best = info_sel.groupby("sale_class")[models].mean().idxmin(axis=1).to_dict()   # per velocity class
print("top-3 on selection set:", top3)
print("per-class choice:", seg_best)

ens = fc[["unique_id", "ds"]].copy()
ens["ENS_mean_all"] = fc[models].clip(lower=0).mean(axis=1)
ens["ENS_median_all"] = fc[models].clip(lower=0).median(axis=1)
ens["ENS_mean_top3"] = fc[top3].clip(lower=0).mean(axis=1)
cls = ens.unique_id.map(info["sale_class"])
ens["SEL_by_class"] = [fc.at[i, seg_best.get(c, top3[0])] for i, c in zip(fc.index, cls)]

top-3 on selection set: ['NHITSx', 'NHITS_noexog', 'LightGBM']
per-class choice: {'A': 'NHITSx', 'B': 'LightGBM', 'C': 'NHITSx', 'D': 'NHITSx', 'E': 'LightGBMx_mono', 'F': 'NHITSx', 'G': 'NHITSx', 'H': 'NHITSx', 'I': 'NHITSx', 'J': 'NHITSx'}


In [9]:
eval_mask = ~ens.unique_id.isin(sel_ids)
both = fc[["unique_id", "ds"] + models].merge(ens, on=["unique_id", "ds"])
eval_scores = bc.summarise(bc.score(both[eval_mask.values], test, train), both[eval_mask.values], test)
eval_scores[["model", "mean_rmsse", "median_rmsse", "wape"]].head(12)

,model,mean_rmsse,median_rmsse,wape
0,ENS_mean_top3,1.245683,0.548550,0.303081
1,NHITS_noexog,1.309712,0.630993,0.375486
2,NHITS_MQ,1.326832,0.642560,0.351931
3,ENS_mean_all,1.342459,0.627299,0.368203
4,NHITS_tuned,1.348833,0.646089,0.415379
5,ENS_median_all,1.354599,0.631003,0.375176
6,LightGBM,1.358644,0.604623,0.352275
7,NHITS,1.362280,0.659863,0.398311
8,LightGBMx_mono,1.392040,0.631458,0.392232
9,LightGBMx,1.392283,0.639369,0.386886


### Reading the result

![Lab 07 results](slides/img/res_lab07.png)

* On the held-out half, the plain **average of the top 3** (NHITSx, NHITS_noexog, LightGBM) beats every single
  model: ≈ 1.25 against ≈ 1.31 for the best single model on that half (NHITS_noexog), about 5 % better. Their
  errors partly cancel: NHITSx is strong on erratic and lumpy series, NHITS_noexog on intermittent ones.
* Picking the **best model per class** did not carry over (≈ 1.40, worse than several single models):
  it chose NHITSx for almost every class, and NHITSx's tail on intermittent series came with it. Selection
  overfits; averaging does not.
* The leaderboard below scores the ensembles on **all** series, half of which chose them, so their leaderboard
  numbers (≈ 1.03 for the top-3 average) are optimistic. Quote the held-out comparison.

*Reference run: 2,000 series, laptop CPU, 500 steps. Your numbers will differ a little; the reading should not.*

Ensembles go on the leaderboard scored on **all** series, like everything else. The evaluation-set table above
is the fair comparison; check that the ranking agrees.

In [10]:
bc.save_forecasts(ens, "07_ensembles")
bc.record(ens, test, train, lab="07 ensembles",
          notes=f"top3={top3}; selection on half the series")

,model,mean_rmsse,median_rmsse,mean_mase,mean_smape,n_series,wape,lab,fit_seconds,notes,recorded_at
0,ENS_mean_top3,1.033705,0.547166,2.403787,0.261779,1996,0.271514,07 ensembles,None,"top3=['NHITSx', 'NHITS_noexog', 'LightGBM']; s...",2026-11-15 17:43:01
1,SEL_by_class,1.120556,0.562684,2.731427,0.261637,1996,0.274597,07 ensembles,None,"top3=['NHITSx', 'NHITS_noexog', 'LightGBM']; s...",2026-11-15 17:43:01
2,ENS_mean_all,1.124495,0.620848,2.595835,0.280294,1996,0.332277,07 ensembles,None,"top3=['NHITSx', 'NHITS_noexog', 'LightGBM']; s...",2026-11-15 17:43:01
3,ENS_median_all,1.142283,0.626187,2.635047,0.283964,1996,0.334944,07 ensembles,None,"top3=['NHITSx', 'NHITS_noexog', 'LightGBM']; s...",2026-11-15 17:43:01


## 5. Final standings

In [11]:
board = bc.leaderboard()
cols = ["model", "lab", "mean_rmsse", "median_rmsse", "wape", "fit_seconds"]
board[cols].round(3)

,model,lab,mean_rmsse,median_rmsse,wape,fit_seconds
1,ENS_mean_top3,07 ensembles,1.034,0.547,0.272,NaN
2,SEL_by_class,07 ensembles,1.121,0.563,0.275,NaN
3,ENS_mean_all,07 ensembles,1.124,0.621,0.332,NaN
4,NHITS_noexog,03 exogenous,1.131,0.623,0.334,32.064
5,ENS_median_all,07 ensembles,1.142,0.626,0.335,NaN
6,NHITS_MQ,06 uncertainty,1.151,0.643,0.329,NaN
7,NHITSx,03 exogenous,1.153,0.563,0.269,36.044
8,LightGBM,00 day1 reference,1.169,0.601,0.310,1.786
9,NHITS_tuned,02 HPO,1.174,0.647,0.357,539.309
10,NHITS,01 GFM default,1.193,0.660,0.352,36.303


In [12]:
fig, ax = plt.subplots(figsize=(10, 0.35 * len(board) + 1))
colors = [palette[2] if str(l).startswith("07") else palette[1] if str(l).startswith(("0", "day2")) and not str(l).startswith("00")
          else palette[0] for l in board["lab"]]
ax.barh(board["model"][::-1], board["mean_rmsse"][::-1], color=colors[::-1])
snaive = board.loc[board.model == "SeasonalNaive", "mean_rmsse"]
if len(snaive):
    ax.axvline(snaive.iloc[0], color="grey", ls=":", label="SeasonalNaive")
    ax.legend()
ax.set_xlabel("mean RMSSE (lower is better)")
ax.set_title("Day 2 final standings  ·  black = Day 1 reference, blue = Day 2 models, orange = ensembles")
plt.tight_layout()
plt.savefig(bc.RESULTS_DIR / "final_standings.png", dpi=150)

### This morning's questions, answered

| Question from the recap | What today's numbers say (reference run) |
|---|---|
| Is the GBDT strong because it is **global** or because of **covariates**? | **Covariates**: LightGBM without them falls to ≈ 1.33, no better than Naive. NHITS without covariates (≈ 1.13) already beats LightGBM (≈ 1.17) |
| Can a neural net learn the hand-built features? | **Mostly yes**, from the raw window; covariates still help it on most series, just not on intermittent ones |
| What does it cost? | 25–35 s per neural fit vs ≈ 2 s; tuning ≈ 540 s for 1.6 % |
| Does it **respond to price** the right way? | **Direction yes, size timid**: promo lift +39 % to +55 % vs +82 % raw in the data (an upper bound) |
| Best of the day? | The **average of the top 3**: on the held-out half ≈ 1.25 vs ≈ 1.31 for the best single model and ≈ 1.36 for Day 1's LightGBM |

## A decision framework to take home

1. **Baselines first.** If you cannot beat SeasonalNaive / ETS on most series, stop and look at the data.
2. **Global before deep.** A GBDT with lags and covariates is the bar. A neural model has to beat it by more
   than the seed-to-seed noise (Lab 02), on more than one window, to justify its training and serving cost.
3. **Tune the few things that matter** (lookback, learning rate / steps, scaler, loss) and stop.
4. **Check responsiveness** before accuracy if the forecast drives pricing or promotion decisions.
5. **Foundation models** are a strong zero-shot baseline; fine-tune only when you have the data and the
   inference budget, and it beats your tuned global model on validation.
6. **Reconcile** when several levels are consumed; pick the method by the level that matters.
7. **Report win rates and segment results**, not only a mean, and **ensemble** when no single model dominates.